In [1]:
import pandas as pd
import numpy as np

In [2]:
numeric_cols = ['amt',
            'lat', 'long', 
            'city_pop', 'merch_lat', 
            'merch_long', 'age',
            'month_sin', 'month_cos']

catagorical_features = ['state', 'category']

In [3]:
from pathlib import Path
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR/"data"

df = pd.read_csv(DATA_DIR/"credit_card_transactions.csv")


In [4]:
df['state'].unique()

<ArrowStringArray>
['NC', 'WA', 'ID', 'MT', 'VA', 'PA', 'KS', 'TN', 'IA', 'WV', 'FL', 'CA', 'NM',
 'NJ', 'OK', 'IN', 'MA', 'TX', 'WI', 'MI', 'WY', 'HI', 'NE', 'OR', 'LA', 'DC',
 'KY', 'NY', 'MS', 'UT', 'AL', 'AR', 'MD', 'GA', 'ME', 'AZ', 'MN', 'OH', 'CO',
 'VT', 'MO', 'SC', 'NV', 'IL', 'NH', 'SD', 'AK', 'ND', 'CT', 'RI', 'DE']
Length: 51, dtype: str

In [5]:
df["category"].unique()

<ArrowStringArray>
[      'misc_net',    'grocery_pos',  'entertainment',  'gas_transport',
       'misc_pos',    'grocery_net',   'shopping_net',   'shopping_pos',
    'food_dining',  'personal_care', 'health_fitness',         'travel',
      'kids_pets',           'home']
Length: 14, dtype: str

In [6]:
def generate_transaction(df):
    # --------------------------------
    # Adding extra columns
    # --------------------------------
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    df['dob'] = pd.to_datetime(df['dob'])

    df['age'] = ((df['trans_date_trans_time'] - df['dob']).dt.days / 365.25).astype(int)

    df['month'] = df['trans_date_trans_time'].dt.month

    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # --------------------------------
    # Decide whether transaction is fraud
    # --------------------------------

    is_fraud = np.random.random() < 0.02

    # Start from realistic transaction
    row = df.sample(1).iloc[0].copy()

    # --------------------------------
    # Normal variation
    # --------------------------------

    row["amt"] *= np.random.normal(1.0, 0.15)
    row["amt"] = max(1, row["amt"])

    row["age"] += np.random.randint(-2, 3)
    row["age"] = np.clip(row["age"], 18, 100)

    row["city_pop"] *= np.random.normal(1.0, 0.05)
    row["city_pop"] = max(1, int(row["city_pop"]))

    # --------------------------------
    # Month
    # --------------------------------

    month = np.random.randint(1, 13)

    row["month_sin"] = np.sin(
        2 * np.pi * month / 12
    )

    row["month_cos"] = np.cos(
        2 * np.pi * month / 12
    )

    # --------------------------------
    # FRAUD MODIFICATIONS
    # --------------------------------

    if is_fraud:

        # 70% chance of unusually high amount
        if np.random.random() < 0.70:
            row["amt"] *= np.random.uniform(1.5, 3.0)

        # 60% chance of unusual merchant location
        if np.random.random() < 0.60:
            row["merch_lat"] = (
                row["lat"] +
                np.random.uniform(-2, 2)
            )

            row["merch_long"] = (
                row["long"] +
                np.random.uniform(-2, 2)
            )

        # 30% chance of changing category
        if np.random.random() < 0.30:
            row["category"] = np.random.choice(
                df["category"].unique()
            )

    # --------------------------------
    # Return
    # --------------------------------
    features = ['category', 'amt', 'state',
                 'lat', 'long', 'city_pop', 
                 'merch_lat', 'merch_long', 'age', 
                 'month_sin', 'month_cos']
    row = row[features]
    return row, is_fraud

In [7]:
samples = []
for _ in range(10):
    row, is_fraud = generate_transaction(df)
    row["is_fraud"] = is_fraud
    samples.append(row)

synth_data = pd.DataFrame(samples)
synth_data.head(5)

,category,amt,state,lat,long,city_pop,merch_lat,merch_long,age,month_sin,month_cos,is_fraud
443044,home,75.060565,CA,38.5234,-120.6763,742,37.859183,-121.101603,32,-2.449294e-16,1.000000e+00,False
774054,shopping_pos,6.324175,LA,30.2385,-90.8435,10189,30.268098,-90.585983,42,-1.000000e+00,-1.836970e-16,False
399170,kids_pets,94.333437,PA,41.4622,-79.1306,4207,42.022091,-78.928151,59,-2.449294e-16,1.000000e+00,False
405294,shopping_pos,1.231738,PA,40.0369,-75.0664,1463930,39.216909,-74.870130,35,5.000000e-01,8.660254e-01,False
90313,grocery_pos,145.219112,IA,42.8511,-93.6200,2940,43.496396,-94.206683,56,-1.000000e+00,-1.836970e-16,False


In [8]:
df.columns

Index(['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long', 'is_fraud', 'merch_zipcode', 'age', 'month',
       'month_sin', 'month_cos'],
      dtype='str')